### Stakaholder Analysis

This notebook is looking at the cleaned datasets to get an overview what would be interesting to further investigate.
The data is picked up after `data.py` stores it in `data/processed`.

Data is processed to answer the following questions with one chart each:<br>
    - How big is the customer base and how did it grow?<br>
    - Does a customer spend more over time or is growth only new customers?<br>
    - Where does the money go?<br>
    - How different are customers?<br>
    - How many customers are real users?<br>
    - Card in store or online?<br>
    - Where is the card used?<br>
    - Which currencies?<br>
    - Seasonality?<br>
    - Who are the top counterparts overall?<br>

 Further we want got get a first insight on:
    - Churn at a glance

The figures are exported to `fiures/eda`, and a short summary for the presentation to be fiound at the very end.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# repo root, whether the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import plots, style
from src.data import load_customer_table, load_sow, load_sow_counterpart, monthly_active
from src.features import build_customer_features

style.apply()
pd.set_option("display.width", 200)

sow = load_sow()
cp = load_sow_counterpart()
feats = build_customer_features(load_customer_table())

monthly = sow.groupby("date").agg(spend=("total_amount", "sum")).join(monthly_active(sow))
monthly["spend_per_active"] = monthly["spend"] / monthly["active_customers"]

ax = plots.active_customers(monthly)
plots.save_fig(ax.figure, "01_active_customers", "eda")

#### monthly active customers interpretation
YPEAL was still growing between January 2021 and December 2023. Starting with around 690 active customers, over 3 years te number of active customers increased by factor 2.24.
After a very strong start beginning of 2021, many customers churned during summer - the reasons therefor may be many but only speculative at that stage.

In [ ]:
monthly_spend = sow.groupby("date").agg(spend=("total_amount", "sum")).join(monthly_active(sow))
monthly_spend["spend_active"] = monthly_spend["spend"] / monthly_spend["active_customers"]

fig, ax = plt.subplots(figsize=(8, 3.4))

ax.plot(monthly_spend.index, monthly_spend["spend_active"], color=style.C[0], lw=2)
for x in (monthly_spend.index[0], monthly_spend.index[-1]):
    val = monthly_spend.loc[x, "spend_active"]
    ax.annotate(
        f"{val:.0f}",
        (x, val),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        color=style.INK2,
    )

plots.month_axis(ax)
style.clean(ax, "spend of all active customer", "CHF / month")

plots.save_fig(fig, "02_spend_active", "eda")
plt.show()

#### spend of all active customer interpretation
to be done